# Workshop — Introdução à IA Responsável na Prática

**Público:** graduação em Ciência da Computação · **Formato:** 3 encontros de 1h40 · **Tema:** viés e *fairness* em Machine Learning

| Encontro | Tema | Seções deste notebook |
|---|---|---|
| **Dia 1** | Treinando um modelo e enxergando de onde vêm os problemas | Seções 1.1 a 1.8 |
| **Dia 2** | Investigando o modelo e os trade-offs de fairness | Seções 2.1 a 2.6 |
| **Dia 3** | Relatório de impacto e apresentações | Seções 3.1 a 3.3 |

### Como usar este notebook
- Execute as células **em ordem** (Shift+Enter).
- Células marcadas com 🔧 **DECISÃO** pedem que o grupo **edite uma variável** antes de executar. Não existe resposta certa única — cada escolha tem consequências, e vocês vão vê-las.
- Células marcadas com ✏️ **TAREFA** trazem perguntas para discutir e anotar.
- Se algo travar, peça ajuda: existe uma configuração de referência (a que já vem preenchida) que sempre funciona.

> **Preparação do ambiente** (uma única vez): `pip install pandas scikit-learn matplotlib`

In [ ]:
# Setup — apenas execute esta célula
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (9, 4)
print("✅ Bibliotecas carregadas. Bom workshop!")

### ⚙️ Infraestrutura do workshop — apenas execute
A célula abaixo define os **6 estudos de caso** disponíveis (fontes e carregamento). Não é preciso ler o código agora — vamos usá-la como uma "biblioteca" do workshop. Cada dataset tenta ser baixado da internet; se não houver conexão, é usada uma cópia local da pasta `dados/`. O material de apoio de cada caso é baixado do repositório do workshop.

In [ ]:
# ----------------------------------------------------------------
# Infraestrutura: fontes de dados, carregadores e contexto de cada caso
# ----------------------------------------------------------------
PASTA_LOCAL = "dados"

# 1ª URL: repositório do workshop; 2ª: fonte original (fallback)
REPO = "https://raw.githubusercontent.com/DiogoCampanha/workshop-ia-responsavel/main"
URLS = {
    "adult":    [f"{REPO}/dados/adult.csv",
                 "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"],
    "compas":   [f"{REPO}/dados/compas.csv",
                 "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"],
    "german":   [f"{REPO}/dados/german.csv",
                 "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data"],
    "diabetes": [f"{REPO}/dados/diabetes.csv",
                 "https://raw.githubusercontent.com/fairlearn/talks/main/2021_scipy_tutorial/data/diabetic_preprocessed.csv"],
    "law":      [f"{REPO}/dados/law.csv",
                 "https://raw.githubusercontent.com/tailequy/fairness_dataset/main/experiments/data/law_school_clean.csv"],
    "student":  [f"{REPO}/dados/student.csv",
                 "https://raw.githubusercontent.com/arunk13/MSDA-Assignments/master/IS607Fall2015/Assignment3/student-por.csv"],
}

def _ler(chave, **kw):
    """Tenta baixar o CSV; se falhar, usa cópia local em dados/<chave>.csv"""
    for url in URLS[chave]:
        try:
            df = pd.read_csv(url, **kw)
            print(f"(dados baixados de {url.split('/')[2]})")
            return df
        except Exception:
            pass
    caminho = os.path.join(PASTA_LOCAL, chave + ".csv")
    if os.path.exists(caminho):
        print(f"(sem internet — usando cópia local: {caminho})")
        return pd.read_csv(caminho, **kw)
    raise RuntimeError(f"Não foi possível carregar '{chave}'. Verifique a internet ou coloque o arquivo em {caminho}")

# Material de apoio dos casos (mantido fora do notebook, no repositório do workshop)
URL_MATERIAL = f"{REPO}/material_instrutor.csv"

def _carregar_material():
    for origem in [URL_MATERIAL, "material_instrutor.csv"]:
        try:
            m = pd.read_csv(origem)
            d = {r["dataset"]: dict(r) for _, r in m.iterrows()}
            for v in d.values():
                v["positivo_e_beneficio"] = str(v["positivo_e_beneficio"]).strip() == "True"
            return d
        except Exception:
            pass
    raise RuntimeError("Material de apoio do workshop não encontrado — avise o instrutor.")

MATERIAL = _carregar_material()

# ---------------- carregadores (um por dataset) ----------------

def carregar_adult():
    nomes = ["idade","classe_trabalho","fnlwgt","educacao","educacao_anos","estado_civil",
             "ocupacao","relacao_familiar","raca","sexo","ganho_capital","perda_capital",
             "horas_semana","pais_origem","renda"]
    df = _ler("adult", header=None, names=nomes, na_values="?", skipinitialspace=True)
    df["alvo"] = (df["renda"].astype(str).str.contains(">50K")).astype(int)
    df = df.drop(columns=["renda","fnlwgt","educacao_anos"])
    return df

def carregar_compas():
    df = _ler("compas")
    df = df[(df["days_b_screening_arrest"] <= 30) & (df["days_b_screening_arrest"] >= -30)]
    df = df[(df["is_recid"] != -1) & (df["c_charge_degree"] != "O")]
    df = df[df["race"].isin(["African-American", "Caucasian"])]
    cols = ["sex","age","race","juv_fel_count","juv_misd_count","juv_other_count",
            "priors_count","c_charge_degree","two_year_recid"]
    df = df[cols].rename(columns={
        "sex":"sexo","age":"idade","race":"raca","juv_fel_count":"crimes_juvenis_graves",
        "juv_misd_count":"delitos_juvenis_leves","juv_other_count":"outros_reg_juvenis",
        "priors_count":"antecedentes","c_charge_degree":"grau_da_acusacao","two_year_recid":"alvo"})
    return df.reset_index(drop=True)

def carregar_german():
    nomes = ["conta_corrente","prazo_meses","historico_credito","proposito","valor_credito",
             "poupanca","tempo_emprego","perc_renda_comprometida","estado_civil_sexo",
             "outros_devedores","anos_na_residencia","patrimonio","idade","outros_financiamentos",
             "moradia","qtd_creditos_no_banco","tipo_emprego","qtd_dependentes","tem_telefone",
             "trabalhador_estrangeiro","risco"]
    df = _ler("german", sep=r"\s+", header=None, names=nomes)
    df["conta_corrente"] = df["conta_corrente"].map(
        {"A11":"saldo negativo","A12":"saldo baixo","A13":"saldo bom","A14":"sem conta"})
    df["historico_credito"] = df["historico_credito"].map(
        {"A30":"nenhum credito","A31":"pagou em dia","A32":"pagando em dia",
         "A33":"atrasos no passado","A34":"conta critica"})
    df["poupanca"] = df["poupanca"].map(
        {"A61":"<100","A62":"100-500","A63":"500-1000","A64":">1000","A65":"sem poupanca"})
    df["tempo_emprego"] = df["tempo_emprego"].map(
        {"A71":"desempregado","A72":"<1 ano","A73":"1-4 anos","A74":"4-7 anos","A75":">7 anos"})
    df["sexo"] = df["estado_civil_sexo"].map(
        {"A91":"masculino","A92":"feminino","A93":"masculino","A94":"masculino","A95":"feminino"})
    df = df.drop(columns=["estado_civil_sexo"])
    df["faixa_etaria"] = np.where(df["idade"] < 25, "jovem (<25)", "25 anos ou mais")
    df["alvo"] = (df["risco"] == 1).astype(int)   # 1 = bom pagador
    df = df.drop(columns=["risco"])
    return df

def carregar_diabetes():
    df = _ler("diabetes")
    df = df[df["race"].isin(["Caucasian", "AfricanAmerican"])]
    df["alvo"] = (df["readmit_30_days"].astype(str) == "True").astype(int)
    df = df.drop(columns=["readmitted","readmit_binary","readmit_30_days"])
    if len(df) > 20000:   # amostra para agilizar o treino em sala
        df = df.sample(20000, random_state=SEED)
    df = df.rename(columns={"race":"raca","gender":"genero","age":"faixa_etaria"})
    return df.reset_index(drop=True)

def carregar_law():
    df = _ler("law")
    df = df.rename(columns={"race":"raca","male":"masculino","lsat":"nota_LSAT","ugpa":"nota_graduacao",
                            "fam_inc":"renda_familiar","fulltime":"tempo_integral","tier":"nivel_faculdade",
                            "decile1b":"decil_1o_ano","decile3":"decil_3o_ano",
                            "zfygpa":"gpa_1o_ano_padronizado","zgpa":"gpa_final_padronizado"})
    df["alvo"] = df["pass_bar"].astype(int)
    df = df.drop(columns=["pass_bar"])
    return df

def carregar_student():
    df = _ler("student", sep=";")
    df["alvo"] = (df["G3"] < 10).astype(int)   # 1 = em risco de reprovar
    df = df.drop(columns=["G1","G2","G3"])     # sem as notas parciais (queremos prever ANTES)
    df["zona"] = df["address"].map({"U":"urbana","R":"rural"})
    df = df.drop(columns=["address"])
    return df

# ---------------- registro dos casos ----------------
CONFIG = {
    "adult": carregar_adult, "compas": carregar_compas, "german": carregar_german,
    "diabetes": carregar_diabetes, "law": carregar_law, "student": carregar_student,
}

def ficha(chave):
    c = MATERIAL[chave]
    print("=" * 78)
    print(c["titulo"])
    print("=" * 78)
    print(f"O modelo prevê ....... {c['preve']}")
    print(f"Previsão positiva .... {c['positivo']}")
    print("-" * 78)
    print("Contexto:", c["contexto"])
    print("=" * 78)

print("✅ Infraestrutura pronta. Datasets disponíveis:", ", ".join(CONFIG.keys()))

---
# 📅 DIA 1 — Treinando um modelo e enxergando de onde vêm os problemas

**Tema do dia:** todo modelo de ML é uma sequência de decisões humanas — e cada decisão molda o resultado.

**Roteiro:** 1.1 escolher o caso → 1.2 conhecer os dados → 1.3 preparar os dados (decisão!) → 1.4 escolher e treinar o modelo (decisão!) → 1.5 avaliar → 1.6 **discutir: onde este modelo pode ser injusto?** → 1.7 a revelação → 1.8 retreinar sem o atributo sensível (decisão!).

## 1.1 · 🔧 DECISÃO 1 — Escolha do estudo de caso do grupo

Cada grupo trabalha com **um** dos 6 casos abaixo durante os 3 dias. Todos são dados **reais**, com problemas reais — e cada um esconde um tipo **diferente** de viés e um dilema ético **diferente**. Mas atenção: **descobrir onde mora o problema é tarefa de vocês**, não do notebook. 🕵️

| Código | Tema | O modelo prevê... |
|---|---|---|
| `"adult"` | 💰 Renda (censo EUA) | quem ganha >US$ 50 mil/ano |
| `"compas"` | ⚖️ Justiça criminal (EUA) | quem vai reincidir em 2 anos |
| `"german"` | 🏦 Crédito bancário (Alemanha) | quem é bom pagador |
| `"diabetes"` | 🏥 Saúde (130 hospitais EUA) | quem volta ao hospital em 30 dias |
| `"law"` | 🎓 Exame da ordem (EUA) | quem passa no bar exam |
| `"student"` | 🏫 Reprovação escolar (Portugal) | quem está em risco de reprovar |

**Edite a variável abaixo** com o código do caso do seu grupo e execute.

In [ ]:
# 🔧 DECISÃO 1 — escolha o dataset do grupo
DATASET = "adult"      # opções: "adult", "compas", "german", "diabetes", "law", "student"

cfg = MATERIAL[DATASET]
df = CONFIG[DATASET]()
ficha(DATASET)
print(f"\n📦 {len(df)} linhas × {df.shape[1]-1} atributos carregados.")

## 1.2 · Conhecendo os dados

Antes de treinar qualquer coisa: **olhe os dados**. Cada linha é uma **pessoa real** (ou uma internação, uma matrícula...). A coluna `alvo` é o que queremos prever (1 = previsão positiva, cujo significado você viu na ficha acima).

> 💡 Muitos problemas de IA Responsável são visíveis **antes** de qualquer treino — basta saber onde olhar.

In [ ]:
# Uma amostra dos dados
df.head(8)

In [ ]:
# Raio-X do dataset
print(f"Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas\n")
print("Distribuição do alvo:")
print(df["alvo"].value_counts(normalize=True).rename({1:"positivo (1)", 0:"negativo (0)"}).round(3).to_string())
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0]
print("\nValores faltantes por coluna:" if len(faltantes) else "\nSem valores faltantes. 🎉")
if len(faltantes):
    print(faltantes.to_string())

### ✏️ TAREFA 1 — leitura crítica (5 min)
1. O alvo é balanceado (perto de 50/50) ou desbalanceado? O que isso implica para a "acurácia" como métrica?
2. Cada linha desta tabela é uma **pessoa**. Se esse sistema fosse usado de verdade, o que aconteceria com quem recebe a previsão positiva? E a negativa?
3. Olhando a lista de colunas: alguma delas te deixa desconfortável como critério de decisão? Por quê? *(anotem — vamos voltar nisso no fim do dia)*

## 1.3 · 🔧 DECISÃO 2 — Preparação dos dados

Uma decisão "técnica" que já é uma decisão **ética**: o que fazer com valores faltantes?

- `"remover"` → descarta linhas incompletas. Cuidado: dados raramente faltam por acaso — se faltam mais para um tipo de pessoa, você está **apagando pessoas** desse tipo (viés de exclusão).
- `"preencher"` → completa com mediana (números) e valor mais comum (categorias). Ninguém é descartado, mas inventamos valores "típicos" — e o "típico" é ditado pela maioria.

In [ ]:
# 🔧 DECISÃO 2 — edite e execute
TRATAMENTO_FALTANTES = "remover"   # "remover" ou "preencher"

dados = df.copy()
antes = len(dados)
if TRATAMENTO_FALTANTES == "remover":
    dados = dados.dropna()
    print(f"Linhas removidas por dados faltantes: {antes - len(dados)} ({(antes-len(dados))/antes:.1%} da base)")
else:
    for col in dados.columns:
        if dados[col].isna().any():
            dados[col] = dados[col].fillna(
                dados[col].median() if dados[col].dtype != "object" else dados[col].mode()[0])
    print("Valores faltantes preenchidos (mediana/moda).")

X_bruto = dados.drop(columns=["alvo"])
X = pd.get_dummies(X_bruto, drop_first=False)        # categorias viram colunas 0/1
print(f"\nMatriz final: {X.shape[0]} pessoas × {X.shape[1]} variáveis (após one-hot encoding)")

## 1.4 · 🔧 DECISÃO 3 — Escolha e treino do modelo

**Como um modelo aprende?** Ele procura, nos exemplos do passado, padrões estatísticos que separam a classe 1 da classe 0 — ajustando parâmetros para **minimizar o erro médio**. Duas consequências importantes:

1. Se o passado foi injusto, o padrão injusto **é exatamente o que o modelo aprende** ("o modelo não sabe o que é justiça, só o que é frequente").
2. Erro **médio** significa que errar muito num grupo pequeno quase não pesa — a minoria é estatisticamente "barata" de ignorar.

| Opção | Modelo | Característica |
|---|---|---|
| `"logistica"` | Regressão Logística | linear, simples, fácil de interpretar |
| `"arvore"` | Árvore de Decisão | regras "se... então", muito interpretável |
| `"floresta"` | Random Forest | conjunto de árvores, geralmente mais preciso e menos interpretável |

Separamos **30% dos dados para teste**: o modelo nunca os vê no treino — é nossa simulação de "pessoas novas chegando ao sistema".

In [ ]:
# 🔧 DECISÃO 3 — edite e execute
MODELO = "arvore"     # "logistica", "arvore" ou "floresta"

def novo_modelo(nome):
    return {
        "logistica": LogisticRegression(max_iter=3000),
        "arvore":    DecisionTreeClassifier(max_depth=6, random_state=SEED),
        "floresta":  RandomForestClassifier(n_estimators=120, random_state=SEED, n_jobs=-1),
    }[nome]

# separamos as PESSOAS (índices) em treino/teste — assim qualquer versão do modelo
# usa exatamente as mesmas pessoas, e as comparações ficam justas
idx_tr, idx_te = train_test_split(dados.index, test_size=0.30,
                                  stratify=dados["alvo"], random_state=SEED)
X_tr, X_te = X.loc[idx_tr], X.loc[idx_te]
y_tr = dados.loc[idx_tr, "alvo"].values
y_te = dados.loc[idx_te, "alvo"].values

modelo = novo_modelo(MODELO)
modelo.fit(X_tr, y_tr)
print(f"✅ Modelo '{MODELO}' treinado com {len(X_tr)} pessoas; será avaliado em {len(X_te)} pessoas nunca vistas.")

## 1.5 · Avaliação inicial — o modelo é "bom"?

A **acurácia** (% de acertos) é a métrica mais famosa — e a mais enganosa. Compare sempre com o **chute da maioria** (prever sempre a classe mais comum). E olhe a **matriz de confusão**: os dois tipos de erro (FP e FN) têm consequências muito diferentes para pessoas reais.

In [ ]:
y_prev = modelo.predict(X_te)
acc = accuracy_score(y_te, y_prev)
chute = max(np.mean(y_te), 1 - np.mean(y_te))
print(f"Acurácia do modelo: {acc:.1%}")
print(f"Chute da maioria:   {chute:.1%}   ← se o modelo não supera isso, ele não aprendeu nada útil")

mc = confusion_matrix(y_te, y_prev)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.imshow(mc, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{mc[i, j]:,}", ha="center", va="center",
                fontsize=15, color="black" if mc[i, j] < mc.max()*0.6 else "white")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["prev. negativo", "prev. positivo"])
ax.set_yticklabels(["real negativo", "real positivo"])
ax.set_title("Matriz de confusão (conjunto de teste)")
ax.set_xlabel("O que o modelo disse"); ax.set_ylabel("O que era verdade")
plt.tight_layout(); plt.show()

fp, fn = mc[0, 1], mc[1, 0]
print(f"⚠️ {fp} falsos positivos → {cfg['fp']}")
print(f"⚠️ {fn} falsos negativos → {cfg['fn']}")

## 1.6 · ✏️ DISCUSSÃO — Onde este modelo pode ser injusto?

O modelo está treinado e tem uma acurácia razoável. Hora da pergunta que separa quem *usa* ML de quem *entende* ML:

> **Se este sistema entrasse em produção amanhã, quem poderia ser tratado injustamente por ele? Por quê?**

Sem ferramentas novas, só com o que vocês já viram (as colunas, a matriz de confusão, o contexto do caso), discutam no grupo:
1. Este modelo decide sobre **pessoas**. Que **características dessas pessoas** poderiam levar o sistema a errar mais para umas do que para outras?
2. Olhem a lista de colunas impressa abaixo: quais vocês consideram **legítimas** para essa decisão? Quais são **eticamente suspeitas**? Alguma pode ser um **atalho disfarçado** para uma característica suspeita (ex.: CEP costuma "entregar" renda e raça)?
3. Escrevam a **hipótese do grupo** na célula seguinte. Não vale pular: cientista que não registra hipótese antes de ver o resultado está só confirmando o que quer. 😉

In [ ]:
# As variáveis que o SEU modelo está usando agora:
print(f"O modelo decide: {cfg['preve']}")
print(f"\nAs {X_bruto.shape[1]} colunas disponíveis para o modelo:\n")
for c in X_bruto.columns:
    exemplo = dados[c].dropna().unique()[:4]
    print(f"  • {c:<28} exemplos: {list(exemplo)}")

In [ ]:
# ✏️ Registrem a hipótese do grupo ANTES da revelação (será comparada no Dia 2!)
HIPOTESES_DO_GRUPO = {
    "colunas_suspeitas":         [],   # ex.: ["idade", "..."] — colunas que NÃO deveriam pesar na decisão
    "quem_pode_ser_prejudicado": "",   # que tipo de pessoa este sistema pode tratar pior?
    "por_que":                   "",   # qual o mecanismo? (poucos dados? histórico injusto? atalhos?)
}
assert HIPOTESES_DO_GRUPO["quem_pode_ser_prejudicado"] != "", "⛔ Preencham a hipótese antes de seguir!"
print("Hipótese registrada ✅ — agora chamem o instrutor para receber o atributo sensível do caso.")

## 1.7 · 🎁 A revelação — o atributo sensível do seu caso

Cada caso deste workshop tem um **atributo sensível** definido: uma característica protegida (por lei ou por ética) que **não deveria** determinar o resultado — e que historicamente determina. **O instrutor vai informar qual é o do seu grupo.** Preencham abaixo com o nome exato da coluna e comparem com a hipótese que registraram.

In [ ]:
# ⬇️ preencham com o nome EXATO da coluna informada pelo instrutor (ex.: "sexo")
SENSIVEL = ""

assert SENSIVEL in dados.columns, f"⛔ Coluna '{SENSIVEL}' não existe. Colunas disponíveis: {list(dados.columns)}"
grupos_todos = dados[SENSIVEL].astype(str)     # o grupo de cada pessoa (usaremos muito no Dia 2)
gr_tr = grupos_todos.loc[idx_tr]
gr_te = grupos_todos.loc[idx_te]

print(f"Atributo sensível do caso: '{SENSIVEL}'")
print(f"\nQuantas pessoas em cada grupo:")
print(grupos_todos.value_counts().to_string())
print(f"\nTaxa de alvo=1 REAL em cada grupo (nos dados):")
print(dados.groupby(grupos_todos)["alvo"].mean().round(3).to_string())
acertou = SENSIVEL in HIPOTESES_DO_GRUPO["colunas_suspeitas"]
print("\n🔎 O grupo tinha listado essa coluna como suspeita? " + ("SIM — boa intuição!" if acertou else "NÃO — discutam: por que ela passou despercebida?"))

## 1.8 · 🔧 DECISÃO 4 — Incluir ou excluir o atributo sensível?

A reação mais comum: *"então é só **remover** essa coluna que o modelo fica justo!"* Será?

Essa estratégia tem nome na literatura — ***fairness through unawareness*** — e vocês vão testá-la **agora**, treinando uma segunda versão do modelo sem a coluna sensível e comparando. Guardem a conclusão: ela volta com força no Dia 2.

- `USAR_ATRIBUTO_SENSIVEL = True` → modelo oficial do grupo **usa** a coluna (transparente, porém explícito).
- `USAR_ATRIBUTO_SENSIVEL = False` → modelo oficial do grupo **não vê** a coluna ("cego").

*(A coluna continua guardada fora do modelo para auditoria — sem ela não conseguiríamos nem medir injustiça.)*

In [ ]:
# 🔧 DECISÃO 4 — edite e execute (pode alternar True/False e re-executar para comparar)
USAR_ATRIBUTO_SENSIVEL = True        # True ou False

# versão COM o atributo (a que já treinamos) vs versão SEM (treinada agora)
X_sem = pd.get_dummies(X_bruto.drop(columns=[SENSIVEL]), drop_first=False)
Xs_tr, Xs_te = X_sem.loc[idx_tr], X_sem.loc[idx_te]
modelo_sem = novo_modelo(MODELO)
modelo_sem.fit(Xs_tr, y_tr)

acc_com = accuracy_score(y_te, modelo.predict(X_te))
acc_sem = accuracy_score(y_te, modelo_sem.predict(Xs_te))
print(f"Acurácia COM '{SENSIVEL}': {acc_com:.2%}")
print(f"Acurácia SEM '{SENSIVEL}': {acc_sem:.2%}")
print(f"Diferença: {abs(acc_com-acc_sem):.2%} — " +
      ("quase nada muda. Suspeito, não? Se a coluna 'não fazia falta', que outras colunas carregam a mesma informação?"
       if abs(acc_com-acc_sem) < 0.01 else "houve mudança. O que o modelo perdeu (ou ganhou) ao ficar 'cego'?"))

# define o MODELO OFICIAL do grupo (usado no Dia 2) conforme a decisão:
if not USAR_ATRIBUTO_SENSIVEL:
    X, X_tr, X_te, modelo = X_sem, Xs_tr, Xs_te, modelo_sem
    print(f"\n➡️ Modelo oficial do grupo: SEM a coluna '{SENSIVEL}'.")
else:
    print(f"\n➡️ Modelo oficial do grupo: COM a coluna '{SENSIVEL}'.")
y_prev = modelo.predict(X_te)

### ✏️ TAREFA 2 — fechamento do Dia 1
1. Sua acurácia superou o chute da maioria por quanto? Isso é muito ou pouco?
2. No **seu caso**, qual erro é pior: FP ou FN? Para **quem** ele é pior?
3. A acurácia quase não mudou ao remover o atributo sensível. Isso prova que o modelo é justo — ou só que ele tem **outros caminhos** para chegar à mesma conclusão? Como vocês poderiam verificar?
4. Liste as 4 decisões que vocês tomaram hoje. Alguma delas foi "neutra"?
5. **Provocação para o Dia 2:** essa acurácia é a média de todo mundo. Será que ela é igual para os grupos do atributo sensível que vocês acabaram de receber? 🤔

---
> **🚀 Mini-projeto (apresentação no Dia 3):** seu grupo produzirá um *relatório de impacto* deste sistema: qual viés encontraram, onde ele nasce, que mitigação tentaram e **o que ela custou**. Tudo o que fizermos no Dia 2 alimenta esse relatório — anotem as descobertas (inclusive a hipótese de hoje: acertaram?).

---
# 📅 DIA 2 — Investigando o modelo e os trade-offs de fairness

**Tema do dia:** auditar o modelo do Dia 1 em ordem crescente de profundidade — dos dados às métricas de fairness — até esbarrar no conflito inevitável entre elas.

**Roteiro:** 2.1 representação → 2.2 desempenho por grupo → 2.3 taxa de seleção e regra dos 80% → 2.4 o que o modelo usa → 2.5 o conflito entre métricas → 2.6 o dilema ético do seu caso.

> ⚠️ Execute as células do Dia 1 antes (Kernel ▸ Restart & Run All até aqui) — inclusive preenchendo `SENSIVEL` (seção 1.7) e a Decisão 4 —, para o modelo estar treinado.

## 2.1 · Representação — quem está nos dados?

**Viés de amostragem:** se um grupo é raro nos dados, o modelo aprende pouco sobre ele (e erra mais). **Viés histórico:** mesmo bem representado, um grupo pode carregar rótulos que refletem desigualdades do passado. A primeira auditoria é uma simples contagem.

In [ ]:
cont = grupos_todos.value_counts()
taxa_pos_real = dados.groupby(grupos_todos)["alvo"].mean().reindex(cont.index)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(cont.index.astype(str), cont.values, color="#4878CF")
axes[0].set_title(f"Quantas pessoas por grupo ({SENSIVEL})")
axes[0].set_ylabel("nº de pessoas")
axes[1].bar(taxa_pos_real.index.astype(str), taxa_pos_real.values, color="#EE854A")
axes[1].set_title("Taxa de alvo=1 REAL por grupo (dados históricos)")
axes[1].set_ylabel("proporção com alvo = 1")
for ax in axes: ax.tick_params(axis="x", rotation=10)
plt.tight_layout(); plt.show()

print(cont.to_string())
print()
for g, t in taxa_pos_real.items():
    print(f"Nos dados, {t:.1%} do grupo '{g}' tem alvo = 1")

### ✏️ TAREFA 3
1. Algum grupo é bem menor que o outro? Que consequência isso teve no aprendizado?
2. As taxas **reais** de alvo=1 já diferem entre grupos. Três explicações possíveis — qual(is) vale(m) para o seu caso?
   - (a) diferença real e legítima entre grupos;
   - (b) desigualdade histórica capturada nos dados;
   - (c) problema de **medição** do rótulo (o rótulo não mede o que a gente pensa que mede).
3. Se a diferença já vem dos dados, o modelo tem culpa de reproduzi-la? E quem a usa?

## 2.2 · Desempenho por subgrupo — o modelo erra igual para todos?

A acurácia global esconde desigualdades. Vamos abrir as métricas por grupo:

- **TPR** (*taxa de verdadeiros positivos*): dos que **são** alvo=1, quantos o modelo reconhece? TPRs iguais entre grupos = **igualdade de oportunidade** (*equal opportunity*).
- **FPR** (*taxa de falsos positivos*): dos que **não são** alvo=1, quantos o modelo marca errado? TPR **e** FPR iguais = ***equalized odds***.

*(Estes nomes são os conceitos formais da literatura — referência para aprofundar: Hardt et al., 2016.)*

In [ ]:
def metricas_por_grupo(y_true, y_pred, grupos):
    """Métricas de desempenho e fairness, calculadas grupo a grupo."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); grupos = np.asarray(grupos)
    linhas = []
    for g in sorted(pd.unique(grupos)):
        m = grupos == g
        yt, yp = y_true[m], y_pred[m]
        vp = int(((yt == 1) & (yp == 1)).sum()); fp = int(((yt == 0) & (yp == 1)).sum())
        vn = int(((yt == 0) & (yp == 0)).sum()); fn = int(((yt == 1) & (yp == 0)).sum())
        linhas.append({
            "grupo": g, "n": int(m.sum()),
            "acurácia":     (vp + vn) / max(len(yt), 1),
            "taxa_seleção": (vp + fp) / max(len(yt), 1),
            "TPR":          vp / max(vp + fn, 1),
            "FPR":          fp / max(fp + vn, 1),
            "precisão":     vp / max(vp + fp, 1),
        })
    return pd.DataFrame(linhas).set_index("grupo").round(3)

tabela = metricas_por_grupo(y_te, y_prev, gr_te)
display(tabela)

ax = tabela[["TPR", "FPR", "acurácia"]].plot.bar(figsize=(9, 3.8), rot=10,
        color=["#6ACC64", "#D65F5F", "#4878CF"])
ax.set_title("O modelo trata os grupos igual? (TPR, FPR e acurácia por grupo)")
ax.set_ylabel("taxa"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

print(f"Lembrete do seu caso — previsão positiva significa: {cfg['positivo']}")

## 2.3 · Taxa de seleção, *disparate impact* e a regra dos 80%

**Taxa de seleção** = % do grupo que recebe previsão **positiva**. A razão entre a menor e a maior taxa é o ***disparate impact ratio***. Na legislação trabalhista dos EUA, uma razão **abaixo de 0,8** ("regra dos 80%", EEOC 1978) é indício de discriminação — taxas de seleção iguais correspondem ao conceito de **paridade demográfica**.

> No seu caso a previsão positiva é benefício ou dano? Isso muda completamente a leitura da barra abaixo.

In [ ]:
sel = tabela["taxa_seleção"]
razao = sel.min() / max(sel.max(), 1e-9)

ax = sel.plot.bar(color="#956CB4", rot=10, figsize=(7.5, 3.5))
ax.set_title("Taxa de seleção por grupo (% que recebe previsão POSITIVA)")
ax.set_ylabel("taxa de seleção")
plt.tight_layout(); plt.show()

print(f"Disparate impact ratio (menor/maior): {razao:.2f}")
print("Regra dos 80%: " + ("✅ PASSA (razão ≥ 0,80)" if razao >= 0.8 else "❌ VIOLADA (razão < 0,80) — indício de impacto desproporcional"))
print(f"\nInterpretação no seu caso: previsão positiva = {cfg['positivo']}")
if not cfg["positivo_e_beneficio"]:
    print("🚨 ATENÇÃO: neste caso a previsão positiva PREJUDICA a pessoa — grupo com MAIOR taxa de seleção é o mais penalizado!")

## 2.4 · Quais variáveis o modelo usa? (porta de entrada da explicabilidade)

Se o modelo se apoia no atributo sensível — ou em variáveis fortemente correlacionadas a ele (***proxies***: CEP ↔ raça, altura ↔ sexo...) — a discriminação entra mesmo "sem querer". Vamos inspecionar a importância das variáveis.

*(Para ir além: **LIME** e **SHAP** explicam previsões individuais — ótima referência de pesquisa para o mini-projeto.)*

In [ ]:
if MODELO == "logistica":
    imp = pd.Series(np.abs(modelo.coef_[0]), index=X.columns)
    tipo = "peso absoluto (|coeficiente|)"
else:
    imp = pd.Series(modelo.feature_importances_, index=X.columns)
    tipo = "importância (redução de impureza)"

top = imp.sort_values(ascending=False).head(12)
cores = ["#D65F5F" if c.startswith(SENSIVEL) else "#4878CF" for c in top.index]
ax = top.iloc[::-1].plot.barh(color=cores[::-1], figsize=(8.5, 4.5))
ax.set_title(f"12 variáveis mais usadas pelo modelo — {tipo}")
plt.tight_layout(); plt.show()

usadas = [c for c in imp.index if c.startswith(SENSIVEL) and imp[c] > 0]
print(f"Variáveis derivadas de '{SENSIVEL}' com importância > 0: {usadas if usadas else 'nenhuma (ou atributo excluído na Decisão 4)'}")
print("\n✏️ Alguma variável do top 12 pode ser um PROXY do atributo sensível? Discutam.")

### ✏️ TAREFA 4 — experimento: "cegar" o modelo resolve?
1. Volte à **Decisão 4** (seção 1.8), mude `USAR_ATRIBUTO_SENSIVEL = False`, execute-a e re-execute as células do Dia 2 até aqui.
2. Compare as métricas por grupo (2.2) e a razão de impacto (2.3) antes/depois. Mudou muito?
3. Na maioria dos casos, **quase nada muda** — outras variáveis "entregam" o grupo (codificação redundante). O que isso diz sobre a estratégia de simplesmente apagar a coluna sensível?

*(Quando terminar, volte para `True` e re-execute, para a turma seguir comparável.)*

## 2.5 · 💥 O conflito entre métricas — ajustando o limiar de decisão

**Ponto central do workshop.** O modelo não decide "sim/não": ele gera uma **probabilidade**, e nós escolhemos o **limiar** de corte (padrão 0,5). Podemos usar **limiares diferentes por grupo** para "consertar" uma injustiça — essa é a mitigação por *pós-processamento* (Hardt et al., 2016).

**Sua missão:** edite os limiares abaixo e tente **igualar a taxa de seleção** entre os grupos (paridade demográfica). Depois observe o que aconteceu com TPR, FPR e precisão. Em seguida tente igualar o TPR. Consegue igualar tudo ao mesmo tempo?

In [ ]:
# Probabilidades previstas para o conjunto de teste (base de tudo nesta seção)
probas = modelo.predict_proba(X_te)[:, 1]
NOMES_GRUPOS = sorted(pd.unique(np.asarray(gr_te)))

def prever_com_limiares(probas, grupos, limiares):
    grupos = np.asarray(grupos)
    yp = np.zeros(len(probas), dtype=int)
    for g, t in limiares.items():
        m = grupos == g
        yp[m] = (probas[m] >= t).astype(int)
    return yp

def painel(limiares):
    base = metricas_por_grupo(y_te, (probas >= 0.5).astype(int), gr_te)
    novo = metricas_por_grupo(y_te, prever_com_limiares(probas, gr_te, limiares), gr_te)
    comp = pd.concat({"limiar único 0,50": base, "seus limiares": novo}, axis=1)
    display(comp.round(3))
    for nome, t in [("limiar único 0,50", base), ("seus limiares", novo)]:
        r = t["taxa_seleção"].min() / max(t["taxa_seleção"].max(), 1e-9)
        acc_g = accuracy_score(y_te, (probas >= 0.5).astype(int) if "único" in nome
                               else prever_com_limiares(probas, gr_te, limiares))
        print(f"{nome:>18}: acurácia global {acc_g:.1%} | disparate impact {r:.2f} "
              + ("✅80%" if r >= 0.8 else "❌80%"))

# Como cada métrica reage ao limiar, grupo a grupo:
ts = np.linspace(0.05, 0.95, 46)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for g in NOMES_GRUPOS:
    m = np.asarray(gr_te) == g
    sel_c, tpr_c, fpr_c = [], [], []
    for t in ts:
        yp = (probas[m] >= t).astype(int); yt = y_te[m]
        sel_c.append(yp.mean())
        tpr_c.append(((yt == 1) & (yp == 1)).sum() / max((yt == 1).sum(), 1))
        fpr_c.append(((yt == 0) & (yp == 1)).sum() / max((yt == 0).sum(), 1))
    axes[0].plot(ts, sel_c, label=g); axes[1].plot(ts, tpr_c, label=g); axes[2].plot(ts, fpr_c, label=g)
for ax, titulo in zip(axes, ["taxa de seleção", "TPR", "FPR"]):
    ax.set_title(titulo + " × limiar"); ax.set_xlabel("limiar"); ax.axvline(0.5, ls=":", c="gray")
axes[0].legend(title=SENSIVEL)
plt.tight_layout(); plt.show()
print("👆 Use estas curvas como 'mapa': onde cortar cada grupo para igualar a métrica que você quer?")

In [ ]:
# 🔧 DECISÃO 5 — edite os limiares por grupo e execute (quantas vezes quiser!)
LIMIARES = {g: 0.50 for g in NOMES_GRUPOS}
# Exemplo: LIMIARES = {"Female": 0.35, "Male": 0.55}  (use os nomes exatos impressos abaixo)
print("Grupos:", NOMES_GRUPOS)

painel(LIMIARES)

### ✏️ TAREFA 5 — o que você acabou de descobrir
1. Ao igualar a **taxa de seleção**, o que aconteceu com o TPR e o FPR de cada grupo? E com a acurácia global?
2. Ao igualar o **TPR**, a taxa de seleção se igualou? E a precisão?
3. Anote os limiares "finais" do grupo — eles vão para o relatório do Dia 3.

### 🧠 Por que é impossível agradar todas as métricas?
Quando as **taxas-base** (a % real de alvo=1) diferem entre grupos, é **matematicamente impossível** igualar simultaneamente calibração, TPR e FPR — é o **teorema da impossibilidade do fairness** (Kleinberg et al., 2016; Chouldechova, 2016). Não é limitação do seu modelo nem falta de dados: **é aritmética**.

**Consequência:** "ser justo" exige **escolher** qual definição de justiça priorizar — e essa escolha é **ética e contextual**, não técnica. Quem deve fazê-la? O dev? A empresa? A lei? As pessoas afetadas?

## 2.6 · 🧩 O dilema ético do SEU caso

Cada dataset deste workshop esconde um dilema **diferente**. Execute a célula abaixo para revelar o do seu grupo — e decidam juntos qual opção defender no Dia 3.

In [ ]:
print(MATERIAL[DATASET]["dilema"])

In [ ]:
# 🔧 DECISÃO 6 — registrem a posição do grupo (será usada no relatório do Dia 3)
DECISAO_DO_GRUPO = {
    "opcao_escolhida": "",     # "A", "B", "C" (ou uma proposta própria do grupo)
    "justificativa":   "",     # por que essa opção? que valores pesaram?
    "quem_ganha":      "",     # quem é beneficiado pela escolha
    "quem_perde":      "",     # quem arca com o custo (todo trade-off tem um lado que perde)
    "salvaguardas":    "",     # o que fariam para reduzir o dano do lado que perde
}
print("Decisão registrada ✅  (revisem no início do Dia 3)")

---
# 📅 DIA 3 — Relatório de impacto e apresentações

**Tema do dia:** comunicar criticamente riscos e limitações de um sistema de IA — a habilidade que transforma análise técnica em responsabilidade real.

Empresas e reguladores usam documentos como *Model Cards* (Mitchell et al., 2019) e avaliações de impacto algorítmico. O de vocês, em miniatura, sai da célula abaixo.

> ⚠️ Execute os Dias 1 e 2 antes (com as decisões finais do grupo), inclusive `LIMIARES` e `DECISAO_DO_GRUPO`.

In [ ]:
# 3.1 · Gera o relatório de impacto do grupo (imprime e salva em arquivo)
def gerar_relatorio():
    base  = metricas_por_grupo(y_te, (probas >= 0.5).astype(int), gr_te)
    ajust = metricas_por_grupo(y_te, prever_com_limiares(probas, gr_te, LIMIARES), gr_te)
    di = lambda t: t["taxa_seleção"].min() / max(t["taxa_seleção"].max(), 1e-9)
    L = []
    L.append("=" * 70)
    L.append("RELATÓRIO DE IMPACTO ALGORÍTMICO — " + cfg["titulo"])
    L.append("=" * 70)
    L.append(f"O sistema prevê: {cfg['preve']}")
    L.append(f"Previsão positiva: {cfg['positivo']}")
    L.append(f"Modelo: {MODELO} | atributo sensível no modelo: {USAR_ATRIBUTO_SENSIVEL} | faltantes: {TRATAMENTO_FALTANTES}")
    L.append("")
    L.append("1) VIÉS ENCONTRADO — " + cfg["tipo_de_vies"])
    L.append("")
    L.append("2) MÉTRICAS COM LIMIAR ÚNICO (0,50):")
    L.append(base.round(3).to_string())
    L.append(f"   disparate impact: {di(base):.2f} " + ("(✅ regra 80%)" if di(base) >= .8 else "(❌ regra 80%)"))
    L.append("")
    L.append(f"3) MITIGAÇÃO TENTADA — limiares por grupo: {LIMIARES}")
    L.append(ajust.round(3).to_string())
    L.append(f"   disparate impact: {di(ajust):.2f} " + ("(✅ regra 80%)" if di(ajust) >= .8 else "(❌ regra 80%)"))
    acc_b = accuracy_score(y_te, (probas >= .5).astype(int))
    acc_a = accuracy_score(y_te, prever_com_limiares(probas, gr_te, LIMIARES))
    L.append(f"   acurácia global: {acc_b:.1%} → {acc_a:.1%}")
    L.append("")
    L.append("4) O QUE A MITIGAÇÃO CUSTOU (preencham na apresentação!):")
    L.append("   compare as colunas acima: qual métrica melhorou? qual piorou? para quem?")
    L.append("")
    L.append("5) DECISÃO DO GRUPO:")
    for k, v in DECISAO_DO_GRUPO.items():
        L.append(f"   {k}: {v or '(preencher na Decisão 6, Dia 2)'}")
    L.append("=" * 70)
    texto = "\n".join(L)
    print(texto)
    nome_arq = f"relatorio_impacto_{DATASET}.txt"
    with open(nome_arq, "w", encoding="utf-8") as f:
        f.write(texto)
    print(f"\n💾 Salvo em {nome_arq}")

    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))
    for ax, met in zip(axes, ["taxa_seleção", "TPR", "FPR"]):
        pd.DataFrame({"limiar único": base[met], "com mitigação": ajust[met]}).plot.bar(
            ax=ax, rot=10, color=["#4878CF", "#6ACC64"])
        ax.set_title(met)
    fig.suptitle("Antes × depois da mitigação — figura para a apresentação")
    plt.tight_layout(); plt.show()

gerar_relatorio()

## 3.2 · Roteiro da apresentação (~8 min por grupo)

1. **O sistema** (1 min) — o que prevê, quem usa, o que acontece com quem recebe previsão positiva.
2. **O viés** (2 min) — que disparidade encontraram, em **que etapa nasce** (dados? rótulo? modelo? uso?) e a evidência (números de 2.1-2.4).
3. **A mitigação** (2 min) — o que tentaram (limiares, remover atributo...) e o resultado.
4. **O custo** (2 min) — o trade-off: qual métrica/grupo pagou o preço. **Sejam honestos: mitigação sem custo provavelmente é mitigação mal medida.**
5. **A decisão** (1 min) — a posição do grupo no dilema e as salvaguardas propostas.

**Checklist:** ☐ números concretos (não "melhorou um pouco") · ☐ dizer quem ganha E quem perde · ☐ 1 limitação que o grupo não resolveu · ☐ a figura "antes × depois".

## 3.3 · Para levar para a vida (e referências)

- Acurácia global esconde injustiças — **desagregue por grupo, sempre**.
- Remover a variável sensível **não** resolve (proxies!). Medir exige *ter* a variável.
- Não existe "justo" universal: existe a **escolha** de qual justiça priorizar — ética e contextual.
- Decisões técnicas "neutras" (limpar dados, escolher métrica, cortar limiar) **moldam vidas**.

**Referências para o mini-projeto e além:** Barocas, Hardt & Narayanan — *Fairness and Machine Learning* (fairmlbook.org) · Hardt et al. 2016 (*Equality of Opportunity*) · Kleinberg et al. 2016 e Chouldechova 2016 (impossibilidade) · ProPublica — *Machine Bias* (2016) · Obermeyer et al. 2019 (viés em saúde, *Science*) · Mitchell et al. 2019 (*Model Cards*) · Gebru et al. 2018 (*Datasheets for Datasets*) · Ribeiro et al. 2016 (LIME) · Lundberg & Lee 2017 (SHAP) · bibliotecas: **Fairlearn** (fairlearn.org) e **AIF360**.

🎉 **Obrigado por participarem!**